In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv(r'C:\Users\ASUS\OneDrive\Desktop\Trek_Data.csv')
print(df.shape)
df.head(3)

## Preprocessing

In [ ]:
# Drop unused columns (including any stray index column)
df = df.drop(columns=[c for c in ['Contact or Book your Trip', 'Unnamed: 0'] if c in df.columns])

# Cost
df['Cost_USD'] = (
    df['Cost'].astype(str)
    .str.replace(r'[\n$,USD]', '', regex=True).str.strip()
    .astype(float)
)
df = df.drop(columns=['Cost'])

# Duration
df['Duration_Days'] = df['Time'].str.strip().str.extract(r'(\d+)')[0].astype(int)
df = df.drop(columns=['Time'])

# Max Altitude
df['Max_Altitude_m'] = (
    df['Max Altitude'].str.replace(',', '', regex=False)
    .str.extract(r'(\d+)')[0].astype(int)
)
df = df.drop(columns=['Max Altitude'])

# Trip Grade — ordinal
grade_map = {
    'Light': 1, 'Easy': 2,
    'Easy-Moderate': 3, 'Easy To Moderate': 3, 'Light+Moderate': 3,
    'Moderate': 4, 'Moderate+Demanding': 5, 'Moderate-Hard': 5,
    'Demanding': 6, 'Strenuous': 7, 'Demanding+Challenging': 8
}
df['Trip_Grade_Ordinal'] = df['Trip Grade'].map(grade_map)
df = df.drop(columns=['Trip Grade'])

# Accommodation — one-hot
accom_map = {
    'Hotel/Guesthouse': 'Guesthouse', 'Hotel/Guesthouses': 'Guesthouse',
    'Hotel/Guest Houses': 'Guesthouse', 'Hotel/Teahouse': 'Teahouse',
    'Hotel/Teahouses': 'Teahouse', 'Hotel/Luxury Lodges': 'LuxuryLodge',
    'Hotel/Lodges': 'Lodge', 'Teahouses/Lodges': 'TeahouseLodge'
}
df['Accomodation'] = df['Accomodation'].map(accom_map)
df = pd.concat([df.drop(columns=['Accomodation']),
                pd.get_dummies(df['Accomodation'], prefix='Accom').astype(int)], axis=1)

# Best Travel Time — one-hot
def norm_travel(v):
    v = str(v).strip().rstrip('.')
    v = v.replace('Setpt', 'Sept')
    v = re.sub(r'\s*-\s*', '-', v)
    v = re.sub(r'\s*&\s*', ' & ', v)
    return re.sub(r'\s+', ' ', v).strip()

travel_map = {
    'March-May & Sept-Dec': 'MarMay_SeptDec',
    'April-May & Sept-Nov': 'AprMay_SeptNov',
    'Jan-May & Sept-Dec':   'JanMay_SeptDec',
    'March-May & Sept-Nov': 'MarMay_SeptNov',
    'March-Nov':            'MarNov'
}
df['Best Travel Time'] = df['Best Travel Time'].apply(norm_travel).map(travel_map)
df = pd.concat([df.drop(columns=['Best Travel Time']),
                pd.get_dummies(df['Best Travel Time'], prefix='Travel').astype(int)], axis=1)

# Scale continuous features
cont_cols = ['Duration_Days', 'Trip_Grade_Ordinal', 'Max_Altitude_m']
scaler = StandardScaler()
df[cont_cols] = scaler.fit_transform(df[cont_cols])

print('Done. Shape:', df.shape)

## Train / Test Split

In [ ]:
feature_cols = [c for c in df.columns if c not in ['Trek', 'Cost_USD', 'Duration_Days']]

X = df[feature_cols].values
y_cost     = df['Cost_USD'].values
y_duration = df['Duration_Days'].values

X_train, X_test, yc_train, yc_test = train_test_split(X, y_cost,     test_size=0.2, random_state=42)
_,       _,      yd_train, yd_test = train_test_split(X, y_duration, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]}  Test: {X_test.shape[0]}')

## Cost Prediction

In [ ]:
models = {
    'OLS':   LinearRegression(),
    'Ridge': Ridge(alpha=10),
    'Lasso': Lasso(alpha=10)
}

for name, model in models.items():
    model.fit(X_train, yc_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(yc_test, pred))
    r2   = r2_score(yc_test, pred)
    print(f'{name:6s}  RMSE: ${rmse:.0f}   R²: {r2:.3f}')

## Duration Prediction

In [ ]:
for name, model in models.items():
    model.fit(X_train, yd_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(yd_test, pred))
    r2   = r2_score(yd_test, pred)
    print(f'{name:6s}  RMSE: {rmse:.2f} days   R²: {r2:.3f}')

## Feature Coefficients

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_tr, title in zip(
    axes,
    [yc_train, yd_train],
    ['Ridge Coefficients — Cost', 'Ridge Coefficients — Duration']
):
    model = Ridge(alpha=10).fit(X_train, y_tr)
    coefs = pd.Series(model.coef_, index=feature_cols).sort_values()
    colors = ['#e74c3c' if v < 0 else '#2980b9' for v in coefs]
    ax.barh(coefs.index, coefs.values, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel('Coefficient value')

plt.tight_layout()
plt.show()

## Regularization Path (Ridge — Cost)

In [ ]:
alphas = np.logspace(-1, 4, 60)
coef_path = [Ridge(alpha=a).fit(X_train, yc_train).coef_ for a in alphas]

plt.figure(figsize=(9, 4))
for i, feat in enumerate(feature_cols):
    plt.plot(np.log10(alphas), [c[i] for c in coef_path], label=feat)
plt.xlabel('log₁₀(alpha)')
plt.ylabel('Coefficient')
plt.title('Ridge Regularization Path — Cost')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()